# Galaxy Zoo EDA

Week 1 exploratory data analysis for the Galaxy Morphology Classification MLOps Pipeline.

Goals:
- Count images and matched labels
- Inspect train / validation split
- Check image size distribution
- Visualize sample images
- Inspect 37-dimensional label distributions
- Inspect label correlations
- Check brightness and simple background-noise signals

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

CSV_PATH = ROOT / 'data/external/training_solutions_rev1.csv'
IMG_DIR = ROOT / 'data/processed/rgb_images'
FIG_DIR = ROOT / 'reports/figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
labels = pd.read_csv(CSV_PATH)
target_cols = [col for col in labels.columns if col != 'GalaxyID']
image_paths = sorted(IMG_DIR.glob('*.jpg'))
image_ids = {path.stem for path in image_paths}
matched = labels[labels['GalaxyID'].astype(str).isin(image_ids)].copy()

summary = {
    'label_rows': len(labels),
    'image_files': len(image_paths),
    'matched_rows': len(matched),
    'target_dimensions': len(target_cols),
    'missing_label_values': int(labels[target_cols].isna().sum().sum()),
}
summary

In [ ]:
indices = np.arange(len(matched))
np.random.shuffle(indices)
train_size = int(0.8 * len(indices))
split_summary = pd.DataFrame([
    {'split': 'train', 'count': train_size},
    {'split': 'validation', 'count': len(indices) - train_size},
])
split_summary

In [ ]:
sample_paths = random.sample(image_paths, k=min(16, len(image_paths)))
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for ax, path in zip(axes.flatten(), sample_paths):
    ax.imshow(Image.open(path).convert('RGB'))
    ax.set_title(path.stem, fontsize=8)
    ax.axis('off')
for ax in axes.flatten()[len(sample_paths):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(FIG_DIR / 'sample_image_grid.png', dpi=160)
plt.show()

In [ ]:
sizes = []
brightness = []
background_std = []

for path in image_paths[: min(1000, len(image_paths))]:
    img = Image.open(path).convert('RGB')
    arr = np.asarray(img, dtype=np.float32) / 255.0
    sizes.append(img.size)
    gray = arr.mean(axis=2)
    brightness.append(float(gray.mean()))
    border = max(gray.shape[0] // 10, 1)
    noise_region = np.concatenate([
        gray[:border, :].ravel(), gray[-border:, :].ravel(),
        gray[:, :border].ravel(), gray[:, -border:].ravel(),
    ])
    background_std.append(float(noise_region.std()))

image_quality = pd.DataFrame({
    'width': [w for w, h in sizes],
    'height': [h for w, h in sizes],
    'brightness': brightness,
    'background_std': background_std,
})
image_quality.describe()

In [ ]:
label_means = matched[target_cols].mean().sort_values(ascending=False)
plt.figure(figsize=(12, 5))
label_means.plot(kind='bar')
plt.ylabel('Mean probability')
plt.title('Galaxy Zoo Label Distribution')
plt.tight_layout()
plt.savefig(FIG_DIR / 'label_distribution.png', dpi=160)
plt.show()

low_frequency = label_means[label_means < 0.05]
low_frequency

In [ ]:
corr = matched[target_cols].corr()
plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.xticks(range(len(target_cols)), target_cols, rotation=90, fontsize=6)
plt.yticks(range(len(target_cols)), target_cols, fontsize=6)
plt.title('Label Correlation Heatmap')
plt.tight_layout()
plt.savefig(FIG_DIR / 'label_correlation_heatmap.png', dpi=160)
plt.show()

## Interview Notes

Before training the model, I inspected label distributions, label correlations, image quality signals, and train-validation split size. This matters because Galaxy Zoo labels are probabilistic and sparse for some conditional morphology questions, so aggregate validation loss alone can hide difficult classes.